In [1]:
from pathlib import Path
import shutil

# Find the attached dataset using a known saved file.
matches = list(Path("/kaggle/input").rglob("all_predictions.csv"))

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one all_predictions.csv; found {len(matches)}. "
        "Send me this output."
    )

prediction_file = matches[0]
roots = [parent for parent in prediction_file.parents
    if parent.name == "rvlcdip_baseline"]

if len(roots) != 1:
    raise RuntimeError("Could not identify the dataset folder.")

source = roots[0].parent
destination = Path("/kaggle/working")

items = ["rvlcdip_baseline",
    "raw_quality_feature_ablation_v1",
    "utility_ablation_leave_condition_out_v1",
    "exploratory_utility_budget.csv",]

missing = [name for name in items if not (source / name).exists()]
existing = [name for name in items if (destination / name).exists()]

if missing:
    raise FileNotFoundError(f"Missing from dataset: {missing}")
if existing:
    raise FileExistsError(f"Already in working folder: {existing}")

for name in items:
    src, dst = source / name, destination / name
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print("RESTORED:", dst)

print("\nReady for the saved-results check.")

RESTORED: /kaggle/working/rvlcdip_baseline
RESTORED: /kaggle/working/raw_quality_feature_ablation_v1
RESTORED: /kaggle/working/utility_ablation_leave_condition_out_v1
RESTORED: /kaggle/working/exploratory_utility_budget.csv

Ready for the saved-results check.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

base = Path(
    "/kaggle/working/rvlcdip_baseline/"
    "four_class_original_pilot/"
    "clean_cnn_standard_preprocessing_v1"
)
oof_path = Path(
    "/kaggle/working/utility_ablation_leave_condition_out_v1/"
    "out_of_fold_scores.csv"
)

pred = pd.read_csv(base / "all_predictions.csv", dtype={"id": str})
oof = pd.read_csv(oof_path, dtype={"id": str})

panels = {
    "main": ["clean", "blur_3", "noise_28"],
    "milder": ["clean", "blur_2", "noise_16"],
    "stronger": ["clean", "blur_5", "noise_45"],
    "jpeg_downsample": ["clean", "jpeg_10", "downsample_022"],
}

# Values reported in the paper
expected = {
    "main":            (80.58, 80.08, 77.50),
    "milder":          (82.83, 81.92, 82.08),
    "stronger":        (72.67, 72.08, 70.17),
    "jpeg_downsample": (81.67, 80.83, 80.33),
}

required_pred = {
    "id", "condition", "label", "image_pred",
    "fusion_pred", "predicted_condition"
}
required_oof = {
    "id", "condition", "label", "image_pred",
    "fusion_pred", "image_conf", "full_score"
}

for name, frame, required in [
    ("predictions", pred, required_pred),
    ("utility scores", oof, required_oof),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")
    if frame.duplicated(["id", "condition"]).any():
        raise ValueError(f"{name} contains duplicate source-condition rows")

# Check that both files describe the same cases and predictions.
check = pred[
    ["id", "condition", "label", "image_pred", "fusion_pred"]
].merge(
    oof[["id", "condition", "label", "image_pred", "fusion_pred"]],
    on=["id", "condition"],
    suffixes=("_pred", "_oof"),
    how="outer",
    indicator=True,
    validate="one_to_one",
)

if not (check["_merge"] == "both").all():
    raise ValueError("The prediction and utility files contain different cases.")

for col in ["label", "image_pred", "fusion_pred"]:
    left = check[f"{col}_pred"].astype(str)
    right = check[f"{col}_oof"].astype(str)
    if not left.equals(right):
        raise ValueError(f"Mismatch between files in {col}")

print("Sources:", pred["id"].nunique())
print("Source-condition cases:", len(pred))
print("Prediction files agree on cases and class predictions.")

rows = []

for panel, conditions in panels.items():
    p = pred[pred["condition"].isin(conditions)].copy()
    u = oof[oof["condition"].isin(conditions)].copy()

    if len(p) != 1200 or len(u) != 1200:
        raise ValueError(f"{panel}: expected 1,200 cases in each file")

    # Fixed quality rule: image-only for predicted blur.
    use_ocr = (
        p["predicted_condition"].astype(str).str.lower() != "blur"
    )
    quality_pred = np.where(
        use_ocr, p["fusion_pred"], p["image_pred"]
    )
    quality_acc = 100 * np.mean(quality_pred == p["label"].to_numpy())
    quality_ocr = 100 * use_ocr.mean()

    # Learned and confidence rankings at 40% OCR.
    count = round(0.40 * len(u))
    image_correct = (
        u["image_pred"].to_numpy() == u["label"].to_numpy()
    )
    fusion_correct = (
        u["fusion_pred"].to_numpy() == u["label"].to_numpy()
    )

    def accuracy_for_ranking(key):
        order = np.argsort(np.asarray(key), kind="stable")
        selected = np.zeros(len(u), dtype=bool)
        selected[order[:count]] = True
        return 100 * np.where(
            selected, fusion_correct, image_correct
        ).mean()

    utility_acc = accuracy_for_ranking(-u["full_score"].to_numpy())
    confidence_acc = accuracy_for_ranking(u["image_conf"].to_numpy())

    exp_quality, exp_utility, exp_conf = expected[panel]
    matches = all([abs(quality_acc - exp_quality) < 0.01,
        abs(utility_acc - exp_utility) < 0.01,
        abs(confidence_acc - exp_conf) < 0.01,])

    rows.append({
        "panel": panel,
        "quality_accuracy": quality_acc,
        "quality_ocr_pct": quality_ocr,
        "utility_40pct": utility_acc,
        "confidence_40pct": confidence_acc,
        "matches_paper": matches,
    })

results = pd.DataFrame(rows)
print("\nRECALCULATED RESULTS")
print(results.round(2).to_string(index=False))

if results["matches_paper"].all():
    print("\nPASS: These selected accuracy values match the paper.")
else:
    print("\nCHECK: At least one value differs. Send me the output.")

print(
    "\nScope: this checks calculations from saved predictions and "
    "scores. It does not verify fresh training, OCR, or every table."
)

Sources: 400
Source-condition cases: 3600
Prediction files agree on cases and class predictions.

RECALCULATED RESULTS
          panel  quality_accuracy  quality_ocr_pct  utility_40pct  confidence_40pct  matches_paper
           main             80.58            66.67          80.08             77.50           True
         milder             82.83            66.67          81.92             82.08           True
       stronger             72.67            66.67          72.08             70.17           True
jpeg_downsample             81.67            66.67          80.83             80.33           True

PASS: These selected accuracy values match the paper.

Scope: this checks calculations from saved predictions and scores. It does not verify fresh training, OCR, or every table.


In [3]:
# ============================================================
# UTILITY ROUTER: FEATURE ABLATIONS + HELD-OUT CONDITIONS
# ============================================================

import ast
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# 1. FIND SAVED PREDICTIONS
# ------------------------------------------------------------

base = Path(
    "/kaggle/working/rvlcdip_baseline/"
    "four_class_original_pilot/"
    "clean_cnn_standard_preprocessing_v1"
)

candidates = [base / "all_predictions.csv",
    base / "evaluation_predictions.jsonl",]

input_dir = Path("/kaggle/input")

if input_dir.exists():
    for name in [
        "all_predictions.csv",
        "evaluation_predictions.jsonl",
    ]:
        candidates.extend(input_dir.rglob(name))

path = next(
    (candidate for candidate in candidates if candidate.is_file()),
    None,
)

if path is None:
    raise FileNotFoundError(
        "Could not find all_predictions.csv or "
        "evaluation_predictions.jsonl. Restore one of these "
        "files and rerun."
    )

if path.suffix == ".csv":
    data = pd.read_csv(path, dtype={"id": str})
else:
    data = pd.read_json(
        path,
        lines=True,
        dtype={"id": str},
    )

required = {
    "id",
    "label",
    "condition",
    "predicted_condition",
    "image_pred",
    "fusion_pred",
    "image_conf",
    "image_probabilities",
}

missing = required - set(data.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

data["id"] = data["id"].astype(str)
data["condition"] = data["condition"].astype(str)
data["predicted_condition"] = (
    data["predicted_condition"]
    .astype(str)
    .str.lower()
)

conditions = {
    "clean",
    "blur_2",
    "blur_3",
    "blur_5",
    "noise_16",
    "noise_28",
    "noise_45",
    "jpeg_10",
    "downsample_022",
}

data = data[data["condition"].isin(conditions)].copy()

data = data.sort_values(
    ["id", "condition"]
).reset_index(drop=True)

if data.duplicated(["id", "condition"]).any():
    raise ValueError("Duplicate source-condition records.")

if data["id"].nunique() != 400 or len(data) != 3600:
    raise ValueError(
        "Expected 400 sources and 3,600 source-condition records."
    )

if not data.groupby("id")["condition"].nunique().eq(9).all():
    raise ValueError(
        "At least one source is missing a test condition."
    )

if not data.groupby("id")["label"].nunique().eq(1).all():
    raise ValueError(
        "A source has inconsistent class labels."
    )

unexpected = set(data["predicted_condition"]) - {
    "clean", "blur", "noise"
}

if unexpected:
    raise ValueError(
        f"Unexpected detector predictions: {unexpected}"
    )

print("Loaded:", path)
print("Sources:", data["id"].nunique())
print("Source-condition cases:", len(data))


# ------------------------------------------------------------
# 2. CREATE PRE-OCR FEATURES
# ------------------------------------------------------------

def parse_probabilities(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)

    values = np.asarray(value, dtype=float)

    if values.shape != (4,) or not np.isfinite(values).all():
        raise ValueError(
            "Invalid image probability vector."
        )

    return values


probabilities = np.stack(
    data["image_probabilities"].map(parse_probabilities)
)

confidence = data["image_conf"].to_numpy(dtype=float)

if not np.isfinite(confidence).all():
    raise ValueError("Non-finite image confidence.")

condition_features = pd.get_dummies(
    data["predicted_condition"]
).reindex(
    columns=["clean", "blur", "noise"],
    fill_value=0,
).to_numpy(dtype=float)

# These features are all available before OCR is invoked.
feature_sets = {
    "full": np.column_stack([
        probabilities,
        confidence,
        condition_features,
    ]),

    "no_condition": np.column_stack([
        probabilities,
        confidence,
    ]),

    "condition_only": condition_features,

    "confidence_only": confidence.reshape(-1, 1),
}


# ------------------------------------------------------------
# 3. CREATE SIGNED OCR-UTILITY TARGET
# ------------------------------------------------------------

labels = data["label"].to_numpy(dtype=int)

image_correct = (
    data["image_pred"].to_numpy(dtype=int) == labels
).astype(int)

fusion_correct = (
    data["fusion_pred"].to_numpy(dtype=int) == labels
).astype(int)

# +1: fusion corrects vision
# -1: fusion harms a correct vision prediction
#  0: both have the same correctness
gain = fusion_correct - image_correct

groups = data["id"].to_numpy()

training_conditions = {
    "clean",
    "blur_3",
    "noise_28",
}

main_indices = np.flatnonzero(
    data["condition"].isin(training_conditions).to_numpy()
)


# ------------------------------------------------------------
# 4. TRAIN ON MAIN CONDITIONS, HOLD OUT SOURCE DOCUMENTS
# ------------------------------------------------------------

scores = {
    name: np.full(len(data), np.nan)
    for name in feature_sets
}

fold_number = np.full(len(data), -1, dtype=int)

splitter = GroupKFold(n_splits=5)

splits = splitter.split(
    np.zeros((len(main_indices), 1)),
    gain[main_indices],
    groups[main_indices],
)

for fold, (train_local, test_local) in enumerate(splits, start=1):

    train_indices = main_indices[train_local]

    held_out_sources = np.unique(
        groups[main_indices[test_local]]
    )

    # Score all nine conditions for held-out sources.
    test_indices = np.flatnonzero(
        np.isin(groups, held_out_sources)
    )

    if set(groups[train_indices]) & set(groups[test_indices]):
        raise RuntimeError("Source overlap within a fold.")

    fold_number[test_indices] = fold

    print(
        f"Fold {fold}: "
        f"{len(np.unique(groups[train_indices]))} training sources; "
        f"{len(held_out_sources)} held-out sources."
    )

    for name, features in feature_sets.items():

        model = RandomForestRegressor(
            n_estimators=150,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            features[train_indices],
            gain[train_indices],
        )

        scores[name][test_indices] = model.predict(
            features[test_indices]
        )

if (fold_number < 0).any():
    raise RuntimeError("Some cases did not receive a fold.")

for name, values in scores.items():
    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Missing out-of-fold scores for {name}."
        )


# ------------------------------------------------------------
# 5. EVALUATE MATCHED OCR BUDGETS
# ------------------------------------------------------------

panels = {
    "main": ["clean", "blur_3", "noise_28"],
    "milder": ["clean", "blur_2", "noise_16"],
    "stronger": ["clean", "blur_5", "noise_45"],
    "jpeg_downsample": [
        "clean",
        "jpeg_10",
        "downsample_022",
    ],
}

budgets = [
    0.30,
    0.40,
    0.50,
    2 / 3,
]

held_out_conditions = [
    "blur_2",
    "blur_5",
    "noise_16",
    "noise_45",
    "jpeg_10",
    "downsample_022",
]


def accuracy_from_order(indices, order, count):
    chosen = np.zeros(len(indices), dtype=bool)
    chosen[order[:count]] = True

    correct = np.where(
        chosen,
        fusion_correct[indices],
        image_correct[indices],
    )

    return 100 * correct.mean()


def evaluate_subset(name, subset_conditions):
    indices = np.flatnonzero(
        data["condition"].isin(subset_conditions).to_numpy()
    )

    if len(indices) == 0:
        raise ValueError(f"Empty subset: {name}")

    n = len(indices)

    # Same seeded random rankings at every budget.
    seed = int.from_bytes(
        name.encode("utf-8"),
        "little",
    ) % (2 ** 32)

    rng = np.random.default_rng(seed)

    random_orders = [
        rng.permutation(n)
        for _ in range(20)
    ]

    confidence_order = np.argsort(
        confidence[indices],
        kind="stable",
    )

    # Uses labels: diagnostic upper bound, not deployable.
    oracle_order = np.argsort(
        -gain[indices],
        kind="stable",
    )

    model_orders = {
        model_name: np.argsort(
            -model_scores[indices],
            kind="stable",
        )
        for model_name, model_scores in scores.items()
    }

    rows = []

    for fraction in budgets:
        count = round(n * fraction)

        row = {
            "subset": name,
            "cases": n,
            "ocr_budget_pct": 100 * count / n,
        }

        for model_name, order in model_orders.items():
            row[model_name] = accuracy_from_order(
                indices,
                order,
                count,
            )

        row["confidence_ranking"] = accuracy_from_order(
            indices,
            confidence_order,
            count,
        )

        row["random_mean"] = np.mean([
            accuracy_from_order(indices, order, count)
            for order in random_orders
        ])

        row["oracle"] = accuracy_from_order(
            indices,
            oracle_order,
            count,
        )

        rows.append(row)

    # Fixed quality rule has its own observed OCR budget.
    use_ocr = (
        data.iloc[indices]["predicted_condition"]
        .to_numpy() != "blur"
    )

    quality_accuracy = 100 * np.where(
        use_ocr,
        fusion_correct[indices],
        image_correct[indices],
    ).mean()

    quality_row = {
        "subset": name,
        "cases": n,
        "quality_accuracy_pct": quality_accuracy,
        "quality_ocr_pct": 100 * use_ocr.mean(),
        "image_only_pct": 100 * image_correct[indices].mean(),
        "always_fusion_pct": 100 * fusion_correct[indices].mean(),
    }

    return rows, quality_row


panel_rows = []
quality_rows = []

for panel_name, panel_conditions in panels.items():
    rows, quality = evaluate_subset(
        panel_name,
        panel_conditions,
    )

    panel_rows.extend(rows)
    quality_rows.append(quality)

condition_rows = []

for condition in held_out_conditions:
    rows, _ = evaluate_subset(
        condition,
        [condition],
    )

    condition_rows.extend(rows)


panel_results = pd.DataFrame(panel_rows)
condition_results = pd.DataFrame(condition_rows)
quality_results = pd.DataFrame(quality_rows)


# ------------------------------------------------------------
# 6. SAVE AND PRINT RESULTS
# ------------------------------------------------------------

output = Path(
    "/kaggle/working/"
    "utility_ablation_leave_condition_out_v1"
)

output.mkdir(parents=True, exist_ok=True)

panel_results.to_csv(
    output / "panel_budget_results.csv",
    index=False,
)

condition_results.to_csv(
    output / "held_out_condition_results.csv",
    index=False,
)

quality_results.to_csv(
    output / "fixed_quality_rule_results.csv",
    index=False,
)

oof = data[
    [
        "id",
        "label",
        "condition",
        "predicted_condition",
        "image_pred",
        "fusion_pred",
        "image_conf",
    ]
].copy()

oof["fold"] = fold_number
oof["signed_gain"] = gain

for name, values in scores.items():
    oof[f"{name}_score"] = values

oof.to_csv(
    output / "out_of_fold_scores.csv",
    index=False,
)

print("\n=== FOUR PANELS: MATCHED OCR BUDGETS ===")
print(panel_results.round(2).to_string(index=False))

print("\n=== HELD-OUT CONDITIONS AT 40% OCR ===")
at_40 = condition_results[
    np.isclose(
        condition_results["ocr_budget_pct"],
        40.0,
    )
]
print(at_40.round(2).to_string(index=False))

print("\n=== FIXED QUALITY RULE ===")
print(quality_results.round(2).to_string(index=False))

print("\nSaved to:", output)

print(
    "\nINTERPRETATION: Each router was trained on clean, blur_3 "
    "and noise_28 from other source documents. The remaining "
    "conditions are held out from router training. Clean is "
    "shared across all panels. Rankings are offline, the oracle "
    "uses labels, and these sources were previously examined."
)

Loaded: /kaggle/working/rvlcdip_baseline/four_class_original_pilot/clean_cnn_standard_preprocessing_v1/all_predictions.csv
Sources: 400
Source-condition cases: 3600
Fold 1: 320 training sources; 80 held-out sources.
Fold 2: 320 training sources; 80 held-out sources.
Fold 3: 320 training sources; 80 held-out sources.
Fold 4: 320 training sources; 80 held-out sources.
Fold 5: 320 training sources; 80 held-out sources.

=== FOUR PANELS: MATCHED OCR BUDGETS ===
         subset  cases  ocr_budget_pct  full  no_condition  condition_only  confidence_only  confidence_ranking  random_mean  oracle
           main   1200           30.00 79.08         76.42           74.92            75.33               76.08        73.00   82.83
           main   1200           40.00 79.83         77.75           77.00            76.83               77.50        73.92   82.83
           main   1200           50.00 80.33         77.92           77.75            77.42               78.33        74.91   82.83
      

In [4]:
from pathlib import Path
import hashlib
import platform
import numpy as np
import pandas as pd
import sklearn
import scipy

# Original files in the attached dataset
old_scores = list(Path("/kaggle/input").rglob("out_of_fold_scores.csv"))
old_preds = list(Path("/kaggle/input").rglob("all_predictions.csv"))

if len(old_scores) != 1 or len(old_preds) != 1:
    raise RuntimeError(
        f"Expected one original score file and prediction file; "
        f"found {len(old_scores)} and {len(old_preds)}."
    )

new_scores = Path(
    "/kaggle/working/utility_ablation_leave_condition_out_v1/"
    "out_of_fold_scores.csv"
)
new_preds = Path(
    "/kaggle/working/rvlcdip_baseline/four_class_original_pilot/"
    "clean_cnn_standard_preprocessing_v1/all_predictions.csv"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("Prediction files identical:",
      sha256(old_preds[0]) == sha256(new_preds))

old = pd.read_csv(old_scores[0], dtype={"id": str})
new = pd.read_csv(new_scores, dtype={"id": str})

merged = old.merge(
    new, on=["id", "condition"],
    suffixes=("_original", "_rerun"),
    how="outer", indicator=True, validate="one_to_one"
)

print("Same source-condition cases:", (merged["_merge"] == "both").all())

if "fold_original" in merged and "fold_rerun" in merged:
    print("Same saved fold assignments:",
          (merged["fold_original"] == merged["fold_rerun"]).all())

for score in ["full_score", "no_condition_score",
              "condition_only_score", "confidence_only_score"]:
    a, b = f"{score}_original", f"{score}_rerun"
    if a in merged and b in merged:
        diff = np.abs(merged[a].to_numpy() - merged[b].to_numpy())
        print(f"{score}: changed={np.sum(diff > 1e-12)}, "
              f"max difference={np.nanmax(diff):.8f}")

print("\nCurrent environment:")
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

Prediction files identical: True
Same source-condition cases: True
Same saved fold assignments: False
full_score: changed=3600, max difference=0.27633374
no_condition_score: changed=3600, max difference=0.28730366
condition_only_score: changed=3600, max difference=0.02782668
confidence_only_score: changed=3600, max difference=0.28630292

Current environment:
Python: 3.13.18
NumPy: 2.1.3
pandas: 2.3.3
SciPy: 1.16.3
scikit-learn: 1.6.1


In [5]:
from pathlib import Path
import json
import pandas as pd

old_path = list(Path("/kaggle/input").rglob("out_of_fold_scores.csv"))[0]
new_path = Path(
    "/kaggle/working/utility_ablation_leave_condition_out_v1/"
    "out_of_fold_scores.csv"
)

old = pd.read_csv(old_path, dtype={"id": str})
new = pd.read_csv(new_path, dtype={"id": str})

for name, frame in [("original", old), ("rerun", new)]:
    counts = frame.groupby("id")["fold"].nunique()
    print(f"{name}: sources assigned to more than one fold:",
          int((counts > 1).sum()))

old_folds = old[["id", "fold"]].drop_duplicates("id")
new_folds = new[["id", "fold"]].drop_duplicates("id")

paired = old_folds.merge(
    new_folds, on="id", suffixes=("_original", "_rerun"),
    validate="one_to_one"
)

table = pd.crosstab(
    paired["fold_original"], paired["fold_rerun"]
)
print("\nSOURCE COUNTS: original folds versus rerun folds")
print(table.to_string())

same_groups = (
    table.max(axis=1).sum() == len(paired)
    and table.max(axis=0).sum() == len(paired)
)
print("\nSame source groups, allowing fold renumbering:", same_groups)

print("\nSAVED ENVIRONMENT INFORMATION")
configs = list(Path("/kaggle/input").rglob("run_config.json"))

def show_versions(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            show_versions(value, f"{prefix}.{key}" if prefix else key)
    elif not isinstance(obj, list):
        if any(word in prefix.lower() for word in
               ["version", "python", "numpy", "pandas", "sklearn", "scikit"]):
            print(prefix, "=", obj)

if configs:
    with open(configs[0]) as f:
        show_versions(json.load(f))
else:
    print("No run_config.json found in the attached input.")

original: sources assigned to more than one fold: 0
rerun: sources assigned to more than one fold: 0

SOURCE COUNTS: original folds versus rerun folds
fold_rerun      1   2   3   4   5
fold_original                    
1              13  17  18  15  17
2              18  13  19  13  17
3              15  20  10  21  14
4              16  14  20   9  21
5              18  16  13  22  11

Same source groups, allowing fold renumbering: False

SAVED ENVIRONMENT INFORMATION
versions.numpy = 2.1.3
versions.opencv = 4.14.0
versions.pandas = 2.3.3
versions.pillow = 13.0.0
versions.pytesseract = 0.3.13
versions.python = 3.13.16
versions.sklearn = 1.6.1
versions.tesseract = 5.3.4
versions.torch = 2.11.0+cu130
versions.torchvision = 0.26.0+cu130


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# The original router cell must have run in this session.
needed = ["data", "feature_sets", "gain"]
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run the utility-router cell first. Missing variables: {missing}"
    )

# Read the original saved fold assignments from the attached backup.
matches = list(Path("/kaggle/input").rglob("out_of_fold_scores.csv"))
if len(matches) != 1:
    raise RuntimeError(f"Expected one original score file, found {len(matches)}")

original = pd.read_csv(matches[0], dtype={"id": str})
current = data.reset_index(drop=True).copy()
current["id"] = current["id"].astype(str)
current["_position"] = np.arange(len(current))

if original.groupby("id")["fold"].nunique().max() != 1:
    raise ValueError("An original source appears in more than one fold.")

# Align current features to the original saved row order.
aligned = original[["id", "condition", "fold"]].merge(
    current[["id", "condition", "_position"]],
    on=["id", "condition"],
    how="left",
    validate="one_to_one",
)

if aligned["_position"].isna().any() or len(aligned) != len(current):
    raise ValueError("Original and current cases do not match.")

positions = aligned["_position"].to_numpy(dtype=int)
folds = aligned["fold"].to_numpy()
main = aligned["condition"].isin(
    ["clean", "blur_3", "noise_28"]
).to_numpy()

target = np.asarray(gain)[positions]
rebuilt = original.copy()

print("Retraining routers using the original saved folds...")

for name in [
    "full", "no_condition", "condition_only", "confidence_only"
]:
    features = np.asarray(feature_sets[name])[positions]
    predictions = np.full(len(aligned), np.nan)

    for fold in sorted(np.unique(folds)):
        train = main & (folds != fold)
        test = folds == fold

        model = RandomForestRegressor(
            n_estimators=150,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(features[train], target[train])
        predictions[test] = model.predict(features[test])

    column = f"{name}_score"
    difference = np.abs(
        predictions - original[column].to_numpy(dtype=float)
    )

    rebuilt[column] = predictions
    print(
        f"{column}: max difference from original = "
        f"{difference.max():.12g}"
    )

# Recalculate the main-panel utility result at 40% OCR.
panel = rebuilt[
    rebuilt["condition"].isin(["clean", "blur_3", "noise_28"])
].copy()

count = round(0.40 * len(panel))
order = np.argsort(-panel["full_score"].to_numpy(), kind="stable")
selected = np.zeros(len(panel), dtype=bool)
selected[order[:count]] = True

image_correct = panel["image_pred"].to_numpy() == panel["label"].to_numpy()
fusion_correct = panel["fusion_pred"].to_numpy() == panel["label"].to_numpy()

accuracy = 100 * np.where(selected, fusion_correct, image_correct).mean()
print(f"\nRebuilt main-panel utility accuracy at 40% OCR: {accuracy:.2f}%")
print("Paper value: 80.08%")

output = Path("/kaggle/working/locked_fold_utility_rebuild_v1")
output.mkdir(exist_ok=True)
rebuilt.to_csv(output / "rebuilt_out_of_fold_scores.csv", index=False)
print("Saved separately to:", output)

Retraining routers using the original saved folds...
full_score: max difference from original = 2.22044604925e-16
no_condition_score: max difference from original = 2.22044604925e-16
condition_only_score: max difference from original = 9.71445146547e-17
confidence_only_score: max difference from original = 1.66533453694e-16

Rebuilt main-panel utility accuracy at 40% OCR: 80.08%
Paper value: 80.08%
Saved separately to: /kaggle/working/locked_fold_utility_rebuild_v1


In [7]:
# ============================================================
# REBUILD UTILITY ROUTERS USING ORIGINAL SOURCE FOLDS
# Requires the research backup attached as a Kaggle input.
# No OCR or CNN retraining is performed.
# ============================================================

from pathlib import Path
import ast
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor


# 1. Find original input files.
input_root = Path("/kaggle/input")

def find_one(filename):
    found = list(input_root.rglob(filename))
    if len(found) != 1:
        raise RuntimeError(
            f"Expected one {filename} in /kaggle/input; "
            f"found {len(found)}."
        )
    return found[0]

prediction_path = find_one("all_predictions.csv")
fold_path = find_one("out_of_fold_scores.csv")

pred = pd.read_csv(prediction_path, dtype={"id": str})
original = pd.read_csv(fold_path, dtype={"id": str})

required_pred = {
    "id", "condition", "label", "image_pred", "fusion_pred",
    "image_conf", "image_probabilities", "predicted_condition"
}
required_original = {
    "id", "condition", "fold", "label", "image_pred",
    "fusion_pred", "full_score", "no_condition_score",
    "condition_only_score", "confidence_only_score"
}

for name, frame, required in [
    ("predictions", pred, required_pred),
    ("original scores", original, required_original),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")
    if frame.duplicated(["id", "condition"]).any():
        raise ValueError(f"{name} contains duplicate cases.")

if original.groupby("id")["fold"].nunique().max() != 1:
    raise ValueError("A source appears in more than one saved fold.")

# Preserve the original case order.
data = original[["id", "condition", "fold"]].merge(
    pred,
    on=["id", "condition"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)

if not (data["_merge"] == "both").all():
    raise ValueError("The saved scores and predictions contain different cases.")

data = data.drop(columns="_merge")

if len(data) != len(original):
    raise ValueError("Unexpected number of aligned cases.")

for column in ["label", "image_pred", "fusion_pred"]:
    if not np.array_equal(
        data[column].astype(str).to_numpy(),
        original[column].astype(str).to_numpy()
    ):
        raise ValueError(f"Original and prediction files disagree on {column}.")

print("Sources:", data["id"].nunique())
print("Cases:", len(data))
print("Saved folds:", sorted(data["fold"].unique()))


# 2. Reconstruct the original pre-OCR features.
def parse_probabilities(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    values = np.asarray(value, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        raise ValueError("Invalid visual probability vector.")
    return values

probabilities = np.stack(
    data["image_probabilities"].map(parse_probabilities)
)
confidence = data["image_conf"].to_numpy(dtype=float)

data["predicted_condition"] = (
    data["predicted_condition"].astype(str).str.lower()
)

unexpected = set(data["predicted_condition"]) - {"clean", "blur", "noise"}
if unexpected:
    raise ValueError(f"Unexpected detector labels: {unexpected}")

condition_features = pd.get_dummies(
    data["predicted_condition"]
).reindex(
    columns=["clean", "blur", "noise"],
    fill_value=0
).to_numpy(dtype=float)

feature_sets = {
    "full": np.column_stack([
        probabilities, confidence, condition_features
    ]),
    "no_condition": np.column_stack([
        probabilities, confidence
    ]),
    "condition_only": condition_features,
    "confidence_only": confidence.reshape(-1, 1),
}

image_correct = (
    data["image_pred"].to_numpy() == data["label"].to_numpy()
)
fusion_correct = (
    data["fusion_pred"].to_numpy() == data["label"].to_numpy()
)
gain = fusion_correct.astype(int) - image_correct.astype(int)

main_training_case = data["condition"].isin(
    ["clean", "blur_3", "noise_28"]
).to_numpy()
folds = data["fold"].to_numpy()


# 3. Retrain using the saved source folds.
scores = {}

for name, features in feature_sets.items():
    rebuilt = np.full(len(data), np.nan)

    for fold in sorted(np.unique(folds)):
        train = main_training_case & (folds != fold)
        test = folds == fold

        model = RandomForestRegressor(
            n_estimators=150,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(features[train], gain[train])
        rebuilt[test] = model.predict(features[test])

    if not np.isfinite(rebuilt).all():
        raise ValueError(f"Missing rebuilt scores for {name}.")

    scores[name] = rebuilt
    difference = np.abs(
        rebuilt - original[f"{name}_score"].to_numpy(dtype=float)
    )
    print(f"{name}: maximum score difference = {difference.max():.12g}")


# 4. Recalculate the 40% panel results.
panels = {
    "main": ["clean", "blur_3", "noise_28"],
    "milder": ["clean", "blur_2", "noise_16"],
    "stronger": ["clean", "blur_5", "noise_45"],
    "jpeg_downsample": ["clean", "jpeg_10", "downsample_022"],
}

expected_full = {
    "main": 80.08,
    "milder": 81.92,
    "stronger": 72.08,
    "jpeg_downsample": 80.83,
}

rows = []

for panel_name, conditions in panels.items():
    indices = np.flatnonzero(data["condition"].isin(conditions).to_numpy())
    if len(indices) != 1200:
        raise ValueError(f"{panel_name}: expected 1,200 cases.")

    count = round(0.40 * len(indices))

    def accuracy(key):
        order = np.argsort(np.asarray(key), kind="stable")
        selected = np.zeros(len(indices), dtype=bool)
        selected[order[:count]] = True
        return 100 * np.where(
            selected,
            fusion_correct[indices],
            image_correct[indices],
        ).mean()

    row = {"panel": panel_name}
    for name in feature_sets:
        row[name] = accuracy(-scores[name][indices])

    row["confidence_ranking"] = accuracy(confidence[indices])
    row["full_matches_paper"] = (
        abs(row["full"] - expected_full[panel_name]) < 0.01
    )
    rows.append(row)

results = pd.DataFrame(rows)

print("\nREBUILT RESULTS AT 40% OCR")
print(results.round(2).to_string(index=False))


# 5. Save new outputs separately; original inputs stay unchanged.
output = Path("/kaggle/working/reproducibility_release_v1")
output.mkdir(parents=True, exist_ok=True)

rebuilt_oof = data[
    ["id", "condition", "fold", "label", "image_pred", "fusion_pred"]
].copy()

for name, values in scores.items():
    rebuilt_oof[f"{name}_score"] = values

rebuilt_oof.to_csv(output / "rebuilt_utility_scores.csv", index=False)
results.to_csv(output / "rebuilt_panel_results_40pct.csv", index=False)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

metadata = {
    "prediction_file_sha256": sha256(prediction_path),
    "original_fold_file_sha256": sha256(fold_path),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scope": (
        "Utility routers refitted from cached predictions using "
        "saved source folds; no CNN training or OCR rerun."
    ),
}

with open(output / "rebuild_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved to:", output)
print(
    "Scope: selected utility results only. This does not reproduce "
    "fresh OCR, CNN training, confidence intervals, or every table."
)

Sources: 400
Cases: 3600
Saved folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
full: maximum score difference = 2.22044604925e-16
no_condition: maximum score difference = 2.22044604925e-16
condition_only: maximum score difference = 9.71445146547e-17
confidence_only: maximum score difference = 1.66533453694e-16

REBUILT RESULTS AT 40% OCR
          panel  full  no_condition  condition_only  confidence_only  confidence_ranking  full_matches_paper
           main 80.08         77.67           76.92            77.17               77.50                True
         milder 81.92         80.50           78.33            80.50               82.08                True
       stronger 72.08         69.83           69.00            70.00               70.17                True
jpeg_downsample 80.83         78.58           77.25            78.58               80.33                True

Saved to: /kaggle/working/reproducibility_release_v1
Scope: selected utility results only